# Faz 2 — Veri Temizleme

Faz 1'de aldığımız kararları ve bulguları burada uyguluyoruz: JSON'u yükle, gereksiz kolonları at,
tarih/sahiplik alanlarını sayısal hale getir, liste/sözlük alanlarından özellik türet, başarı
metriklerini hesapla ve temiz veriyi `data/processed/games_clean.parquet` olarak kaydet.

Tekrar eden kod yazmamak için tüm adımlar `src/load.py` ve `src/clean.py` içindeki fonksiyonlarla yapılıyor;
bu notebook onları sırayla çağırıp her adımın etkisini gösteriyor.

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd

from src.load import load_raw_games
from src.clean import (
    MIN_REVIEWS_THRESHOLD,
    drop_duplicate_apps,
    drop_unused_columns,
    parse_release_date,
    parse_estimated_owners,
    add_list_features,
    add_review_features,
    classify_review_label,
    add_success_features,
)
from src.features import multi_hot_encode

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

## 1. Ham Veriyi Yükle

In [2]:
df = load_raw_games('../data/raw/games.json')
print('Ham veri:', df.shape)
df.head(2)

Ham veri: (147352, 43)


,app_id,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,header_image,website,support_url,support_email,windows,mac,linux,metacritic_score,metacritic_url,achievements,recommendations,notes,supported_languages,full_audio_languages,packages,developers,publishers,categories,genres,screenshots,movies,user_score,score_rank,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.00,0,,,,,https://shared.akamai.steamstatic.com/store_it...,,,,True,False,False,0,,0,0,,[],[],[],[],[],[],[],[https://shared.akamai.steamstatic.com/store_i...,[],0,,0,0,0 - 0,0,0,0,0,0,0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,https://shared.akamai.steamstatic.com/store_it...,http://mangagamer.org/supipara,http://mangagamer.com,support@mangagamer.com,True,False,False,0,,0,231,,[English],[],[{'title': 'Buy Supipara - Chapter 1 Spring Ha...,[minori],[MangaGamer],"[Single-player, Steam Trading Cards, Steam Clo...",[Adventure],[https://shared.akamai.steamstatic.com/store_i...,[],0,,252,3,0 - 20000,8,0,8,0,65,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."


## 2. Tekrarlı Kayıtları Temizle

Faz 1'de tekrarlı `app_id` bulunmadığını doğrulamıştık; burada savunma amaçlı yine kontrol ediyoruz.

In [3]:
df = drop_duplicate_apps(df)

drop_duplicate_apps: 0 tekrarlı satır silindi (147,352 -> 147,352)


## 3. Gereksiz Kolonları At

URL/görsel alanları (`header_image`, `screenshots`, `movies`, ...), kullanılmayan metin alanları
(`detailed_description`, `short_description`) ve analiz için ilgisiz alanlar (`discount`,
`full_audio_languages`, `notes`, `packages`, `score_rank`, `metacritic_url`) atılıyor. `about_the_game`
metnini Faz 4.6'daki (opsiyonel) metin analizi için saklıyoruz.

In [4]:
df = drop_unused_columns(df)
print('Kolon sayısı:', df.shape[1])

drop_unused_columns: 14 kolon atıldı -> ['detailed_description', 'short_description', 'header_image', 'website', 'support_url', 'support_email', 'screenshots', 'movies', 'metacritic_url', 'score_rank', 'notes', 'packages', 'full_audio_languages', 'discount']
Kolon sayısı: 29


## 4. Çıkış Tarihini İşle

`release_date` metnini (`"Aug 1, 2023"`) datetime'a çevirip `release_year`, `release_month`,
`release_weekday` türetiyoruz; parse edilemeyen veya gelecek tarihli kayıtları işaretliyoruz.

In [5]:
df = parse_release_date(df)
df[['release_date', 'release_date_parsed', 'release_year', 'release_month', 'release_weekday', 'is_future_release']].head(3)

parse_release_date: 0 tarih parse edilemedi (release_date_parsed = NaT)
parse_release_date: 1 oyun gelecek tarihli (henüz çıkmamış/duyurulmuş)


,release_date,release_date_parsed,release_year,release_month,release_weekday,is_future_release
0,"Aug 1, 2023",2023-08-01,2023,8,Tuesday,False
1,"Jul 29, 2016",2016-07-29,2016,7,Friday,False
2,"May 6, 2019",2019-05-06,2019,5,Monday,False


In [6]:
print('Gelecek tarihli oyun(lar):')
df.loc[df['is_future_release'], ['app_id', 'name', 'release_date']]

Gelecek tarihli oyun(lar):


,app_id,name,release_date
125386,4418940,Roten,"Dec 1, 2026"


## 5. Tahmini Sahip Sayısı Aralığını Sayısala Çevir

`"20000 - 50000"` formatındaki metni `owners_min`, `owners_max` ve orta nokta `owners_mid`'e çeviriyoruz.

In [7]:
df = parse_estimated_owners(df)
df[['estimated_owners', 'owners_min', 'owners_max', 'owners_mid']].drop_duplicates(subset='estimated_owners').sort_values('owners_min').head(12)

parse_estimated_owners: 0 satırda aralık parse edilemedi


,estimated_owners,owners_min,owners_max,owners_mid
0,0 - 0,0.0,0.0,0.0
1,0 - 20000,0.0,20000.0,10000.0
30,20000 - 50000,20000.0,50000.0,35000.0
85,50000 - 100000,50000.0,100000.0,75000.0
12,100000 - 200000,100000.0,200000.0,150000.0
54,200000 - 500000,200000.0,500000.0,350000.0
26,500000 - 1000000,500000.0,1000000.0,750000.0
350,1000000 - 2000000,1000000.0,2000000.0,1500000.0
172,2000000 - 5000000,2000000.0,5000000.0,3500000.0
3388,5000000 - 10000000,5000000.0,10000000.0,7500000.0


## 6. Liste/Sözlük Alanlarından Özellik Türet

`num_languages`, `num_genres`, `num_tags`, `num_categories`, `num_platforms` sayısal özellikleri;
`tags` (sözlük) alanından da sadece etiket adlarını içeren `tags_list` türetiliyor (oy sayıları atılıyor).

In [8]:
df = add_list_features(df)
df[['num_languages', 'num_genres', 'num_tags', 'num_categories', 'num_platforms']].describe()

,num_languages,num_genres,num_tags,num_categories,num_platforms
count,147352.000000,147352.000000,147352.000000,147352.000000,147352.000000
mean,5.504133,2.717852,8.012738,4.586120,1.297492
std,13.677536,1.460714,8.471077,3.422528,0.622804
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,2.000000,0.000000,2.000000,1.000000
50%,1.000000,3.000000,5.000000,4.000000,1.000000
75%,5.000000,4.000000,18.000000,6.000000,1.000000
max,103.000000,19.000000,21.000000,32.000000,3.000000


## 7. Review Özellikleri ve Sınıflandırması

`total_reviews`, `positive_ratio` ve Steam'in review-özet mantığına benzer basitleştirilmiş bir
`review_label` sınıflandırması türetiliyor (eşikler `src/clean.py` içindeki `classify_review_label`
docstring'inde açıklanmıştır).

In [9]:
df = add_review_features(df)
df = classify_review_label(df)
df['review_label'].value_counts()

add_review_features: 64,360 oyunun hiç review'u yok
classify_review_label: {'No Reviews': 64360, 'Not Enough Reviews': 26301, 'Very Positive': 22644, 'Mixed': 13785, 'Positive': 10129, 'Overwhelmingly Positive': 7678, 'Negative': 2095, 'Overwhelmingly Negative': 360}


review_label
No Reviews                 64360
Not Enough Reviews         26301
Very Positive              22644
Mixed                      13785
Positive                   10129
Overwhelmingly Positive     7678
Negative                    2095
Overwhelmingly Negative      360
Name: count, dtype: int64

## 8. Başarı Metrikleri: is_free, is_indie, is_self_published, description_length, log dönüşümleri

**`is_indie` kararı:** `genres` alanı kullanılıyor (tags değil). Faz 1'de görüldüğü üzere `tags`
topluluk oyu gerektirdiği için düşük-review'lu oyunların %43'ünde tamamen boş; bu da tam olarak
incelemek istediğimiz "az bilinen indie oyunlar" grubunda tag-bazlı tanımı sistematik olarak
güvenilmez kılıyor. `genres` ise listeleme anında atanır ve her oyunda mevcuttur.

Sahiplik/review/oynama-süresi gibi çok çarpık dağılan sayısal alanlar için silmeden `log1p`
dönüşümü ekleniyor (Faz 3/4'teki grafiklerde log ölçek olarak kullanılacak).

In [10]:
df = add_success_features(df)
df[['is_free', 'is_indie', 'is_self_published']].mean()

add_success_features: is_indie oranı = 66.8%
add_success_features: is_free oranı   = 19.8%


is_free              0.197873
is_indie             0.667734
is_self_published    0.680948
dtype: float64

## 9. Review Eşiği Kararı: "Yeterli Veri" Filtresi

Dokümanın önerdiği gibi analiz iki seviyede yapılacak: **(1) tüm veri**, **(2) en az
`MIN_REVIEWS_THRESHOLD` review'u olan oyunlar**. Eşik olarak **10** seçildi — bu, Steam'in kendi
"Need More Reviews" eşiğiyle aynı ve aşağıdaki dağılıma göre de mantıklı bir kesim noktası:
10'un altında review'u olan oyunlarda `positive_ratio` çok gürültülü (tek bir review bile oranı
%0 ↔ %100 arası sıçratabiliyor).

In [11]:
print(f"Eşik: >= {MIN_REVIEWS_THRESHOLD} review")
print(f"Toplam oyun               : {len(df):,}")
print(f"Yeterli review'u olan oyun : {df['has_enough_reviews'].sum():,} ({df['has_enough_reviews'].mean():.1%})")
print(f"Hiç review'u olmayan oyun  : {(df['total_reviews'] == 0).sum():,}")
print(f"1-9 arası review'u olan oyun: {((df['total_reviews'] > 0) & (df['total_reviews'] < MIN_REVIEWS_THRESHOLD)).sum():,}")

Eşik: >= 10 review
Toplam oyun               : 147,352
Yeterli review'u olan oyun : 56,691 (38.5%)
Hiç review'u olmayan oyun  : 64,360
1-9 arası review'u olan oyun: 26,301


## 10. Aykırı Değerler (Outlier)

Doküman uyarısı gereği aykırı değerler **silinmiyor**; `owners_mid`, `total_reviews` ve
`average_playtime_forever` gibi çok çarpık dağılan sayısal alanlar için `log1p_*` kolonları zaten
Bölüm 8'de eklendi. Çarpıklığı karşılaştıralım.

In [12]:
from scipy.stats import skew

for col in ['owners_mid', 'total_reviews', 'average_playtime_forever']:
    raw_skew = skew(df[col].dropna())
    log_skew = skew(df[f'log1p_{col}'].dropna())
    print(f"{col:28} ham çarpıklık={raw_skew:8.2f}   log1p çarpıklık={log_skew:6.2f}")

owners_mid                   ham çarpıklık=   78.39   log1p çarpıklık= -0.71
total_reviews                ham çarpıklık=  193.40   log1p çarpıklık=  1.18
average_playtime_forever     ham çarpıklık=  288.19   log1p çarpıklık=  2.02


## 11. En Sık Görülen Türler ve Etiketler (Multi-hot Örneği)

`src/features.py` içindeki `multi_hot_encode` fonksiyonu, Faz 3 ve Faz 5'te tekrar kullanılacak.
Burada sadece doğru çalıştığını ve en sık 10 türü/etiketi doğru bulduğunu gösteriyoruz
(kalıcı olarak `games_clean.parquet`'e eklemiyoruz; her analizde ilgili top-N yeniden hesaplanacak).

In [13]:
genre_encoded, top_genres = multi_hot_encode(df, 'genres', top_n=10)
print('En sık 10 tür:', top_genres)
genre_encoded.mean().sort_values(ascending=False)

En sık 10 tür: ['Indie', 'Casual', 'Action', 'Adventure', 'Simulation', 'Strategy', 'RPG', 'Free To Play', 'Early Access', 'Sports']


genres_indie           0.667734
genres_casual          0.422906
genres_action          0.369170
genres_adventure       0.359506
genres_simulation      0.208460
genres_strategy        0.191623
genres_rpg             0.171372
genres_free_to_play    0.102245
genres_early_access    0.099985
genres_sports          0.038730
dtype: float64

In [14]:
tag_encoded, top_tags = multi_hot_encode(df, 'tags_list', top_n=10)
print('En sık 10 etiket:', top_tags)
tag_encoded.mean().sort_values(ascending=False)

En sık 10 etiket: ['Singleplayer', 'Indie', 'Action', 'Casual', 'Adventure', '2D', '3D', 'Simulation', 'Strategy', 'Puzzle']


tags_list_singleplayer    0.341868
tags_list_indie           0.329646
tags_list_action          0.249946
tags_list_casual          0.248215
tags_list_adventure       0.238986
tags_list_2d              0.181884
tags_list_3d              0.122190
tags_list_simulation      0.121491
tags_list_strategy        0.121227
tags_list_puzzle          0.111135
dtype: float64

## 12. Temizlenmiş Veriyi Kaydet

In [15]:
from pathlib import Path

out_path = Path('../data/processed/games_clean.parquet')
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(out_path, index=False)
print(f"Kaydedildi: {out_path} ({out_path.stat().st_size / 1024**2:,.1f} MB)")
print('Son şekil:', df.shape)

Kaydedildi: ../data/processed/games_clean.parquet (132.3 MB)
Son şekil: (147352, 53)


In [16]:
# Doğrulama: dosyayı geri oku ve birebir aynı şekle sahip olduğunu kontrol et
check = pd.read_parquet(out_path)
assert check.shape == df.shape
print('Parquet round-trip doğrulandı:', check.shape)
check.dtypes

Parquet round-trip doğrulandı: (147352, 53)


app_id                                     int64
name                                         str
release_date                                 str
required_age                               int64
price                                    float64
dlc_count                                  int64
about_the_game                               str
reviews                                      str
windows                                     bool
mac                                         bool
linux                                       bool
metacritic_score                           int64
achievements                               int64
recommendations                            int64
supported_languages                       object
developers                                object
publishers                                object
categories                                object
genres                                    object
user_score                                 int64
positive            

## 13. Özet — Faz 2'de Alınan Kararlar

- **Veri kaynağı:** JSON (Faz 1 kararı), CSV'deki kolon kayması nedeniyle.
- **Tekrarlı kayıt:** 0 bulundu, savunma amaçlı dedup adımı eklendi.
- **Gereksiz 14 kolon** (URL/görsel/ham metin) atıldı; bellek kullanımı düşürüldü.
- **`is_indie`** tanımı için `genres` esas alındı (`tags` yerine) — tags verisi düşük-review'lu
  oyunlarda sistematik olarak eksik.
- **Review eşiği:** `MIN_REVIEWS_THRESHOLD = 10` — Faz 3/4'te analizler hem tüm veri hem bu eşiği
  geçen alt küme üzerinde tekrarlanacak.
- **Aykırı değerler silinmedi**; log1p dönüşümleri eklendi (çarpıklık ham veride çok yüksek,
  log1p'te belirgin şekilde düşüyor — Bölüm 10).
- **`review_label`**, Steam'in gerçek algoritmasının basitleştirilmiş bir versiyonu; eşikler
  `src/clean.py` docstring'inde açık şekilde belirtildi.
- Temiz veri `data/processed/games_clean.parquet`'e kaydedildi ve round-trip doğrulandı.